# AI Travel Operations Agent

## Project Introduction

The AI Travel Operations Agent is an intelligent multi-agent system designed to automate and simplify travel planning.

The system uses Large Language Models (LLMs), LangGraph, and the Model Context Protocol (MCP) to coordinate specialized AI agents.

Each agent performs a specific task, such as transportation research, hotel discovery, weather analysis, activity recommendations, budget estimation, and itinerary verification.

### Project Objectives

1. Understand user travel requirements.
2. Discover suitable transportation options.
3. Find hotels based on budget and preferences.
4. Analyze weather conditions.
5. Recommend tourist attractions and activities.
6. Estimate total travel expenses.
7. Generate a personalized travel itinerary.
8. Verify information using connected data sources.

### Example User Request

"Plan a 3-day trip from Hyderabad to Goa for 2 travelers with a total budget of INR 15,000."

### Expected Output

- Transportation recommendations
- Hotel recommendations
- Weather information
- Tourist attractions
- Day-wise travel itinerary
- Estimated total budget

## Technology Stack

| Technology | Purpose |
|---|---|
| Python | Core development language |
| Jupyter Notebook | Development and experimentation |
| Groq API | LLM inference |
| LangChain | LLM integration |
| LangGraph | Multi-agent workflow orchestration |
| MCP | Standardized tool integration |
| Pydantic | Data validation |
| HTTPX | External API requests |
| FastAPI | Backend API development |
| PostgreSQL | Persistent data storage |
| React | Frontend development |
| Tailwind CSS | User interface styling |

## System Architecture

User Travel Request
        |
        v
Travel Request Validation
        |
        v
LangGraph Orchestrator
        |
        +---- Transport Agent
        |
        +---- Hotel Agent
        |
        +---- Weather Agent
        |
        +---- Activities Agent
        |
        +---- Budget Agent
        |
        +---- Review Agent
        |
        v
External APIs and MCP Tools
        |
        v
Final Travel Itinerary

# Import Required Libraries

In [9]:
import os
import sys
import platform

from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [10]:
print("Project: AI Travel Operations Agent")
print("Python Version:", sys.version.split()[0])
print("Operating System:", platform.system())
print("Working Directory:", Path.cwd())

Project: AI Travel Operations Agent
Python Version: 3.12.4
Operating System: Windows
Working Directory: C:\Users\Rohit singh\Desktop\ai-travel-operations-agent\notebooks


## Environment Configuration

Environment variables are used to store sensitive configuration values, such as API keys.

The GROQ_API_KEY is stored in a .env file instead of being hardcoded in the source code.

In [11]:
PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing.")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [12]:
MODEL_NAME = "openai/gpt-oss-120b"

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized successfully.")
print("Selected Model:", MODEL_NAME)

Groq client initialized successfully.
Selected Model: openai/gpt-oss-120b


# Test LLM Response

In [13]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an intelligent AI Travel Assistant. "
                "Provide clear, practical, and structured travel guidance."
            )
        },
        {
            "role": "user",
            "content": (
                "Introduce yourself and explain how you can "
                "assist users with travel planning."
            )
        }
    ],
    temperature=0.3,
    max_completion_tokens=300
)

answer = response.choices[0].message.content

print("AI Travel Assistant Response:\n")
print(answer)

AI Travel Assistant Response:

**Hello! I’m your AI Travel Assistant – a friendly, knowledgeable companion that helps you plan smooth, enjoyable trips from start to finish.**  

Below is a quick overview of what I can do for you, organized into the main stages of travel planning:

---

## 1️⃣ Destination Discovery  
- **Idea generation:** Suggest destinations based on interests (culture, adventure, food, relaxation, wildlife, etc.).  
- **Best‑time advice:** Highlight optimal travel windows, weather patterns, and local events or festivals.  
- **Travel restrictions:** Provide up‑to‑date visa, vaccination, and entry‑requirement information.

## 2️⃣ Budget & Timeline Planning  
- **Cost estimates:** Break down typical expenses for flights, accommodation, meals, transport, and activities.  
- **Budget tools:** Help you set daily spending limits and suggest ways to stretch your money.  
- **Trip length:** Recommend ideal trip durations for different destinations and travel styles.

## 3️⃣ 

# Create a Reusable LLM Function

In [14]:
def generate_llm_response(
    user_prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant."
) -> str:

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.3,
        max_completion_tokens=500
    )

    return response.choices[0].message.content or ""

In [15]:
result = generate_llm_response(
    user_prompt="Suggest 5 popular tourist attractions in Goa."
)

print(result)

Here are five of the most popular tourist attractions in Goa, each offering a different flavor of the state’s vibrant culture, natural beauty, and history:

| # | Attraction | Why It’s Popular | Quick Tips |
|---|------------|------------------|------------|
| **1** | **Baga & Calangute Beaches** | The iconic “Party Beaches” of North Goa. Golden sand, lively shacks, water‑sports (parasailing, jet‑skiing), and a buzzing nightlife scene. | Arrive early to snag a good spot on the sand; sunset at Baga is spectacular. Dress modestly when moving away from the beach area. |
| **2** | **Fort Aguada & Sinquerim Beach** | A well‑preserved 17th‑century Portuguese fort with a lighthouse offering panoramic sea views. The adjoining Sinquerim Beach is quieter than Baga, perfect for swimming. | Bring a hat and water; the fort’s steps can be steep. Entry is free, but a small parking fee applies if you drive. |
| **3** | **Old Goa (Basilica of Bom Jesus & Se Cathedral)** | A UNESCO World Heritage site s

# Create the Travel Request Schema

In [17]:
# Import Pydantic
from pydantic import BaseModel, Field, model_validator
from datetime import date
from typing import Optional

In [18]:
# Define the Schema

class TravelRequest(BaseModel):
    source: str = Field(
        ...,
        min_length=2,
        description="Starting city of the trip"
    )

    destination: str = Field(
        ...,
        min_length=2,
        description="Destination city"
    )

    start_date: date = Field(
        ...,
        description="Trip start date"
    )

    end_date: date = Field(
        ...,
        description="Trip end date"
    )

    travelers: int = Field(
        default=1,
        ge=1,
        le=20,
        description="Number of travelers"
    )

    budget: float = Field(
        ...,
        gt=0,
        description="Total trip budget in INR"
    )

    currency: str = "INR"

    preferences: list[str] = Field(
        default_factory=list,
        description="User travel preferences"
    )

    @model_validator(mode="after")
    def validate_trip(self):
        if self.end_date < self.start_date:
            raise ValueError(
                "End date must be on or after start date."
            )

        if self.source.strip().casefold() == self.destination.strip().casefold():
            raise ValueError(
                "Source and destination must be different."
            )

        return self

In [19]:
# Test the Schema

travel_request = TravelRequest(
    source="Hyderabad",
    destination="Goa",
    start_date="2026-11-10",
    end_date="2026-11-12",
    travelers=2,
    budget=15000,
    preferences=["Beaches", "Local Food", "Sightseeing"]
)

print(travel_request)

source='Hyderabad' destination='Goa' start_date=datetime.date(2026, 11, 10) end_date=datetime.date(2026, 11, 12) travelers=2 budget=15000.0 currency='INR' preferences=['Beaches', 'Local Food', 'Sightseeing']


In [20]:
travel_data = travel_request.model_dump(mode="json")

print(travel_data)

{'source': 'Hyderabad', 'destination': 'Goa', 'start_date': '2026-11-10', 'end_date': '2026-11-12', 'travelers': 2, 'budget': 15000.0, 'currency': 'INR', 'preferences': ['Beaches', 'Local Food', 'Sightseeing']}


In [21]:
# Create the Shared Agent State

from typing import TypedDict, Optional, Any

class TravelAgentState(TypedDict, total=False):
    travel_request: dict[str, Any]

    transport_options: list[dict[str, Any]]
    hotel_options: list[dict[str, Any]]
    weather_info: dict[str, Any]
    activities: list[dict[str, Any]]

    budget_summary: dict[str, Any]
    review_result: dict[str, Any]

    itinerary: dict[str, Any]
    errors: list[str]

In [22]:
# Initialize the Shared State

initial_state: TravelAgentState = {
    "travel_request": travel_data,
    "transport_options": [],
    "hotel_options": [],
    "weather_info": {},
    "activities": [],
    "budget_summary": {},
    "review_result": {},
    "itinerary": {},
    "errors": []
}

print("Initial Travel Agent State:")
print(initial_state)

Initial Travel Agent State:
{'travel_request': {'source': 'Hyderabad', 'destination': 'Goa', 'start_date': '2026-11-10', 'end_date': '2026-11-12', 'travelers': 2, 'budget': 15000.0, 'currency': 'INR', 'preferences': ['Beaches', 'Local Food', 'Sightseeing']}, 'transport_options': [], 'hotel_options': [], 'weather_info': {}, 'activities': [], 'budget_summary': {}, 'review_result': {}, 'itinerary': {}, 'errors': []}


In [24]:
# Validate Incorrect Input
# Test Validation

try:
    invalid_request = TravelRequest(
        source="Hyderabad",
        destination="Goa",
        start_date="2026-11-12",
        end_date="2026-11-10",
        travelers=2,
        budget=15000
    )
except Exception as error:
    print("Validation Error:")
    print(error)

Validation Error:
1 validation error for TravelRequest
  Value error, End date must be on or after start date. [type=value_error, input_value={'source': 'Hyderabad', '...rs': 2, 'budget': 15000}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


# Groq LLM Integration & Reusable LLM Service

In [26]:
# Define ask_llm()

def ask_llm(
    prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant.",
    temperature: float = 0.3,
    max_tokens: int = 500
) -> str:
    """
    Send a prompt to the Groq LLM and return its response.
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_completion_tokens=max_tokens
    )

    return response.choices[0].message.content or ""

In [27]:
# est the LLM Service

result = ask_llm(
    prompt="Suggest five popular tourist attractions in Goa.",
    system_prompt=(
        "You are a travel planning assistant. "
        "Give clear and concise recommendations."
    )
)

print(result)

**Top 5 Must‑See Attractions in Goa**

| # | Attraction | Why Visit? | Quick Tips |
|---|------------|------------|------------|
| 1 | **Baga & Calangute Beaches** | Iconic sandy stretches with vibrant nightlife, water sports, and beach shacks. | Arrive early for a quieter beach; rent a scooter to explore nearby beaches. |
| 2 | **Basilica of Bom Bom (Our Lady of the Immaculate Conception)** | One of the oldest churches in Goa, famous for its stunning Baroque architecture and the miraculous “miraculous” statue. | Dress modestly (shoulders/knees covered); attend the 6 pm mass for a serene experience. |
| 3 | **Fort Aguada & Lighthouse** | 17th‑century Portuguese fort offering panoramic sea views and a historic lighthouse. | Visit at sunset for the best photo ops; combine with a stop at nearby Sinquerim Beach. |
| 4 | **Dudhsagar Waterfalls** (on the Goa–Karnataka border) | Majestic four‑tiered waterfall set in lush jungle; perfect for trekking and boat rides. | Book a guided jeep trek o

In [28]:
# Generate a Travel Summary

import json

travel_prompt = f"""
Analyze the following travel request.

Travel request:
{json.dumps(travel_data, indent=2)}

Provide:
1. A short summary of the trip.
2. The main travel preferences.
3. Important planning considerations.
4. Information that still needs verification.
"""

travel_summary = ask_llm(
    prompt=travel_prompt,
    system_prompt=(
        "You are an AI travel planning analyst. "
        "Do not invent live prices or availability."
    )
)

print(travel_summary)

**1. Short Summary of the Trip**  
- **Origin:** Hyderabad (India)  
- **Destination:** Goa (India) – a coastal state famous for its beaches, Portuguese‑influenced cuisine, and relaxed vibe.  
- **Dates:** 10 Nov 2026 (arrival) – 12 Nov 2026 (departure) – a 3‑day, 2‑night getaway.  
- **Travelers:** 2 adults.  
- **Budget:** ₹15,000 total (≈ ₹7,500 per person) for transportation, accommodation, meals and activities.  
- **Key Interests:** Beach time, tasting local Goan food, and sightseeing around the region.

**2. Main Travel Preferences**  
| Preference | What it means for the itinerary |
|------------|---------------------------------|
| **Beaches** | Prioritise staying near a popular beach (e.g., Calangute, Baga, Anjuna, or Palolem) and schedule at least a few hours of beach leisure each day. |
| **Local Food** | Include meals at authentic Goan eateries (seafood shacks, “feni” bars, traditional “Goan thali”) and possibly a short food‑tour or cooking demo. |
| **Sightseeing** | Fit 

In [29]:
# Add Basic Error Handling

import logging
from groq import APIConnectionError, APIStatusError, RateLimitError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("travel_agent")


def safe_ask_llm(
    prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant."
) -> str:
    try:
        return ask_llm(
            prompt=prompt,
            system_prompt=system_prompt
        )

    except RateLimitError:
        logger.warning("Groq API rate limit reached.")
        return "The AI service is temporarily rate-limited."

    except APIConnectionError:
        logger.exception("Unable to connect to Groq.")
        return "The AI service is currently unreachable."

    except APIStatusError as error:
        logger.error("Groq API returned status %s.", error.status_code)
        return "The AI service returned an API error."

In [36]:
# Test Error-Handled Function

result = safe_ask_llm(
    "Explain why travel budgets should include emergency expenses."
)

print(result)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


### Why Your Travel Budget Needs an Emergency‑Expense Cushion  

| Reason | What It Means for You | Example |
|--------|----------------------|---------|
| **Unexpected medical needs** | Health emergencies can happen anywhere, and treatment abroad is often far more expensive than at home. | You sprain an ankle while hiking in the Andes and need an urgent X‑ray and physical‑therapy session. |
| **Travel‑related disruptions** | Flights, trains, or ferries can be delayed, canceled, or over‑booked, forcing you to pay for alternate transport or overnight lodging. | A storm shuts down the airport in Bangkok; you need a last‑minute hotel and a new flight. |
| **Lost or stolen belongings** | Replacing passports, visas, credit cards, or essential gear on the road can be costly and time‑consuming. | Your backpack is stolen in Barcelona; you must buy a new phone, charger, and replace your wallet. |
| **Natural disasters or civil unrest** | Sudden events may require you to evacuate, relocate, or e

# Transport Agent

A Transport Agent is a specialized AI component responsible for analyzing transportation requirements and comparing available travel options based on cost, duration, convenience, and user preferences.

In [32]:
# Create Sample Transport Data

transport_data = [
    {
        "mode": "Flight",
        "price_per_person": 4500,
        "duration_hours": 1.5,
        "comfort": 5
    },
    {
        "mode": "Train",
        "price_per_person": 1200,
        "duration_hours": 14,
        "comfort": 4
    },
    {
        "mode": "Bus",
        "price_per_person": 1000,
        "duration_hours": 16,
        "comfort": 3
    }
]

print("Sample transport data created successfully!")

Sample transport data created successfully!


In [34]:
# Calculate Total Transport Cost

def calculate_transport_cost(
    price_per_person: float,
    travelers: int
) -> float:
    return price_per_person * travelers


travelers = travel_data["travelers"]

for option in transport_data:
    option["total_cost"] = calculate_transport_cost(
        option["price_per_person"],
        travelers
    )

transport_data

[{'mode': 'Flight',
  'price_per_person': 4500,
  'duration_hours': 1.5,
  'comfort': 5,
  'total_cost': 9000},
 {'mode': 'Train',
  'price_per_person': 1200,
  'duration_hours': 14,
  'comfort': 4,
  'total_cost': 2400},
 {'mode': 'Bus',
  'price_per_person': 1000,
  'duration_hours': 16,
  'comfort': 3,
  'total_cost': 2000}]

In [37]:
# Rank Transport Options
# Define Ranking Logic

def rank_transport_options(
    options: list[dict],
    budget: float
) -> list[dict]:
    ranked_options = []

    for option in options:
        item = option.copy()

        item["within_budget"] = item["total_cost"] <= budget

        ranked_options.append(item)

    ranked_options.sort(
        key=lambda item: (
            not item["within_budget"],
            item["total_cost"],
            item["duration_hours"]
        )
    )

    return ranked_options

In [39]:
# Run Ranking

budget = travel_data["budget"]

ranked_transport = rank_transport_options(
    transport_data,
    budget
)

for index, option in enumerate(ranked_transport, start=1):
    print(f"\nRank: {index}")
    print("Mode:", option["mode"])
    print("Total Cost:", option["total_cost"])
    print("Duration (hours):", option["duration_hours"])
    print("Within Budget:", option["within_budget"])


Rank: 1
Mode: Bus
Total Cost: 2000
Duration (hours): 16
Within Budget: True

Rank: 2
Mode: Train
Total Cost: 2400
Duration (hours): 14
Within Budget: True

Rank: 3
Mode: Flight
Total Cost: 9000
Duration (hours): 1.5
Within Budget: True


In [40]:
# Use the LLM to Explain Options
# Generate a Transport Recommendation

import json

transport_prompt = f"""
You are a transport planning assistant.

Travel request:
{json.dumps(travel_data, indent=2)}

Transport options:
{json.dumps(ranked_transport, indent=2)}

Compare the options based on total cost, duration, and comfort.

Recommend a suitable option and explain the trade-offs.
Clearly mention that these are sample estimates, not live prices.
Do not invent schedules or availability.
"""

transport_recommendation = ask_llm(
    prompt=transport_prompt,
    system_prompt=(
        "You are a careful travel analyst. "
        "Use only the supplied data for numerical comparisons."
    )
)

print(transport_recommendation)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Comparison of the three sample transport options (Hyderabad → Goa, 2 travelers)**  

| Mode   | Total cost (INR) | Travel duration (hours) | Comfort rating* |
|--------|------------------|--------------------------|-----------------|
| Bus    | 2 000            | 16                       | 3 |
| Train  | 2 400            | 14                       | 4 |
| Flight | 9 000            | 1.5                      | 5 |

\*Comfort is a simple 1‑5 scale provided in the data (5 = most comfortable).

### Trade‑off summary  

| Factor | Bus | Train | Flight |
|--------|-----|-------|--------|
| **Cost** – cheapest option, well under the INR 15 000 budget. | ✔︎ | ✔︎ (slightly higher) | ❌ – still under budget but ~4‑5× the bus cost. |
| **Time** – longest journey; may reduce time available for beaches, food and sightseeing. | ❌ (16 h) | ❌ (14 h) | ✔︎ (only 1.5 h). |
| **Comfort** – basic seating, likely more crowding. | 3/5 | 4/5 (better seats, smoother ride) | 5/5 (most comfortable, minimal fati

In [41]:
# Save Results in Agent State
# Update Shared State

initial_state["transport_options"] = ranked_transport

print("Transport Agent results saved in shared state.")
print(initial_state["transport_options"])

Transport Agent results saved in shared state.
[{'mode': 'Bus', 'price_per_person': 1000, 'duration_hours': 16, 'comfort': 3, 'total_cost': 2000, 'within_budget': True}, {'mode': 'Train', 'price_per_person': 1200, 'duration_hours': 14, 'comfort': 4, 'total_cost': 2400, 'within_budget': True}, {'mode': 'Flight', 'price_per_person': 4500, 'duration_hours': 1.5, 'comfort': 5, 'total_cost': 9000, 'within_budget': True}]
